# Trustworthy RAG — quickstart

Grounded answers with citations, **calibrated abstention instead of bluffing**, and a full audit trace. Runs offline (no weights) so this notebook works as-is; swap in a real model with one line.


In [ ]:
from srag import SelfCorrectingRAG

docs = [
    {'id': 'p1', 'text': 'Refunds are available within 30 days of purchase.'},
    {'id': 'p2', 'text': 'Shipping is free on orders over 50 dollars.'},
    {'id': 'p3', 'text': 'Support hours are 9am to 5pm on weekdays.'},
]
rag = SelfCorrectingRAG.from_documents(docs)   # mode='trustworthy' (default)


## Answer a supported question

In [ ]:
r = rag.ask('What is the refund window?')
print(r.status)        # answered / hedged / abstained
print(r.message)       # answer + citations
print(r.citations)


## Abstain instead of inventing
The corpus doesn't cover this, so a trustworthy system should refuse.

In [ ]:
r = rag.ask('What is the CEO\'s home address?')
print(r.status, '->', r.message)
assert r.abstained


## Inspect the decision trace

In [ ]:
import json
tj = rag.ask('What is the refund window?').trace_json()
print(json.dumps({k: tj[k] for k in ['status','citations','confidence','diagnoses']}, indent=2))
# tj['trace'] is the full ordered pipeline; rag.ask(...).trace_html() renders a page


## Bring your own model
```python
from srag import OpenAIChat   # or OllamaChat, VLLMChat, TransformersChat
rag = SelfCorrectingRAG.from_documents(docs, llm=OpenAIChat('gpt-4o-mini'), real_models=True)
```
Turn on the correction loop with `mode='self_correcting'`. See `docs/USING_YOUR_OWN_LLM.md`.